In [ ]:
#IMPORTS
import pandas as pd
import numpy as np
from skbio.stats.composition import clr
from scipy.stats import pearsonr
from typing import List, Tuple, Optional, Union
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
#FUNCTION DEFINITIONS

#A class for creating a population of sequences with a particular correlation to an input sequence of values
class SubPop():
    # Parameters- Each parameter will either create a single value (int) or a uniform range between two values (Tuple).
    # size: number of observations in the population.
    # slope: The range of correlations the population has with the dependent variable.
    # intercept: The intercept of the population count and the dependent variable.
    # variance: The variance of the population around the value determined by the.
    # zeros: The overall portion of null (0) values present in the data. Must be between 0 and 1.
    def __init(
        self,
        size: Union[int,Tuple[int,int]] = 0,
        slope: Union[int,Tuple[int,int]] = 1,
        intercept: Union[int,Tuple[int,int]] = 0,
        variance: Union[int,Tuple[int,int]] = 1,
        zeros: Union[int,Tuple[int,int]] = 0
    ):
        self._size = size,
        self._slope = slope,
        self._intercept = intercept,
        self._variance = variance

        assert zeros <= 1.0 and zeros >= 0
        self_zeros = zeros
    def _createRange(self, _range: Union[int, Tuple[int,int]]):
        if isinstance(_range,int):
            return _range
        else:
            return np.random.uniform(_range[0],_range[1])
    def size(self):
        return self._createRange(self._size)
    def slope(self):
        return self._createRange(self._slope)
    def intercept(self):
        return self._createRange(self._intercept)
    def variance(self):
        return self._createRange(self._variance)
    def zeros(self):
        return self._createRange(self._zeros)


def simulatePop(
    subPops: List[SubPop],
    input_vars: List[Union[int,float]],
    col_names: Optional[str] = None
):
    """
    Parameters:
    subPops: A List of subpopulations to comprise the whole population
    input_var: The sequence with which the simulated populations will be correlated
    col_names: The optional column names for the resulting data.
    
    Returns: A Pandas Dataframe containing all of the resulting observations, with column names matching the input variables.
    """
    sim_data = []
    for subPop in subPops:
        for i in range(subPop.size):
            dependent_vars = subPop.slope()*input_vars + subPop.intercept()
            dependent_vars += np.random.normal(0,subPop.variance()*dependent_vars.std(),len(dependent_vars))
            sim_data.append(dependent_var)
    if not col_names:
        try:
            pd.DataFrame(np.array(sim_data), columns = [str(var) for var in input_vars.tolist()])
        except:
            pd.DataFrame(np.array(sim_data), columns = [str(var) for var in range(len(input_vars))])
    return pd.DataFrame(np.array(sim_data), columns = col_names)




    
def percPositivePearson(df, alpha: float = .05):
    """
    Parameters:
    df: dataframe containing columns 'p_value' and 'r_value' corresponding to significance and Pearson R correlation
    alpha: P-value threshold to filter array by

    Returns: A float within [0,1] representing the portion of positive Pearson Correlations within the dataframe, and the number of significant observations remaining
    """
    df_alpha = df[df['p_value'] <= alpha]
    pos_r = df_alpha[df_alpha['r_value'] > 0].shape[0]
    sig_r = df_alpha[df_alpha['r_value'] != 0].shape[0]

    ppp= np.nan
    if sig_r > 0:
        ppp = pos_r/sig_r
    return (ppp, sig_r)


def calc_ppp(df, input_vars: Optional[List[Union[int,float]]] = None, max_alpha: float = 1.0): 
    """
    Parameters:
    df[MxN]: dataframe containing various observations which are to be correlated to an input sequence
    input_var[1xN]: the input sequence
    max_alpha: the maximum alpha to look at.

    The input DataFrame must only include the columns for the observations which are correlated to the input sequence.

    Returns: A dataframe containing the % positive Pearson correlation between the observations in df and input_var,
    for each p-value in the range [0.0,1.0,0.01] 
    """
    if ('r_value' not in df.columns) or ('p_value' not in df.columns):
        assert input_vars
        # Calculating PearsonR for each row of df and adding that data to df in columns "r_value" and "p_value"
        df = pd.concat((df,calc_pearson(df,input_vars)),axis=1,join='inner')
        
    # Calculating %PPC for each alpha value
    results = []
    for alpha in np.arange(max_alpha,0.0,-0.01):
        ppp, pop_size = percPositivePearson(df, alpha)
        results += [[alpha, ppp, pop_size]]
    
    # return %PPC and Pop-size df.
    return pd.DataFrame(results,columns=["alpha","ppp", "pop_size"])
    
def graph_ppp(
    df,
    input_vars: Optional[List[Union[int,float]]] = None,
    max_alpha: float = 1.0,
    name: str = "simulated population",
    processed: bool = False,
    include_pop_size: bool = True
):
    """
    Parameters:
    df[MxN]: dataframe containing various observations which are to be correlated to an input sequence
    input_var[1xN]: the input sequence
    max_alpha: the maximum alpha value to graph.
    name: the name of the population being sampled
    processed: Whether or not df has already been transformed into the correct form to be graphed
    include_pop_size: Whether or not to include a background area chart showing the % of overall sample population beating each p-value threshold

    The input DataFrame must only include the columns for the observations which are correlated to the input sequence.

    Displays: A chart showing % positive pearson correlation between the observations in the dataset and the input vars for only
    the observations beating each p-value threshold from 0 to 1.
    """
    if not processed:
        df = calc_ppp(df, input_vars, max_alpha = max_alpha)
    # Plotting %PPC vs. alpha for DF2 (0s dropped)
    fig, ax1 = plt.subplots(figsize=(8,6))
    ax1.scatter(df['alpha'],df['ppp'], s=20, alpha=0.6)
    ax1.axhline(y=0.5, color='red', linestyle='--', linewidth=1.5, label='50% threshold')
    ax1.set_ylim(0,1)
    ax1.set_xlabel("alpha")
    ax1.set_ylabel("% Positive Pearson Correlation")

    if include_pop_size:
        ax2 = ax1.twinx()
        ax2.fill_between(df['alpha'],0,df['pop_size'], alpha=0.2, color='gray', label='No. Observations beating threshold')
        ax2.set_ylabel('No. Observations beating threshold')
        ax2.tick_params(axis='y',labelcolor='gray')
    
    plt.title(f'% Positive Pearson Correlation vs. p-value Threshold ({name})')
    plt.tight_layout()
    plt.show()

def graph_ppp_subplot(
    df,
    input_vars: Optional[List[Union[int,float]]] = None,
    max_alpha: float = 1.0,
    name: str = "simulated population",
    processed: bool = False,
    include_pop_size: bool = True,
    ax: Optional[plt.Axes] = None
):
    """Modified version that accepts an axes object"""
    if not processed:
        df = calc_ppp(df, input_vars, max_alpha=max_alpha)
    
    # Use provided ax or create new one
    if ax is None:
        fig, ax = plt.subplots(figsize=(8,6))
    
    ax.scatter(df['alpha'], df['ppp'], s=20, alpha=0.6)
    ax.axhline(y=0.5, color='red', linestyle='--', linewidth=1.5, label='50% threshold')
    ax.set_ylim(0, 1)
    ax.set_xlabel("alpha")
    ax.set_ylabel("% Positive Pearson Correlation")
    
    if include_pop_size:
        ax2 = ax.twinx()
        ax2.fill_between(df['alpha'], 0, df['pop_size'], alpha=0.2, color='gray', 
                         label='No. Observations beating threshold')
        ax2.set_ylabel('No. Observations beating threshold')
        ax2.tick_params(axis='y', labelcolor='gray')
    
    ax.set_title(f'%PPC vs alpha, {name}')
    
    return ax


def graph_ppp_line_subplot(
    df,
    input_vars: Optional[List[Union[int,float]]] = None,
    max_alpha: float = 1.0,
    name: str = "simulated population",
    processed: bool = False,
    include_pop_size: bool = True,
    ax: Optional[plt.Axes] = None
    ):
    if not processed:
        df = calc_ppp(df, input_vars, max_alpha=max_alpha)
    
    # Use provided ax or create new one
    if ax is None:
        fig, ax = plt.subplots(figsize=(8,6))
    
    ax.plot(df['alpha'], df['ppp'], alpha=0.6, label=name)
    
    if include_pop_size:
        ax2 = ax.twinx()
        ax2.fill_between(df['alpha'], 0, df['pop_size'], alpha=0.2, color='gray')
        ax2.set_yticks([])
    
    return ax

In [ ]:
#CONSTANTS
VOLTAGES = [0, 200, 400, 600, 800]

In [ ]:
#READ DATA
complete = pd.read_excel("MLEA_111724_otu_table_complete.xlsx", index_col=0)
dtype_mapping = {
    'numeric': float,
    'categorical': str
}
dtypes = dict(zip(complete.columns.tolist(), [dtype_mapping[dtype] for dtype in complete.iloc[0]]))
complete = pd.read_excel("MLEA_111724_otu_table_complete.xlsx", index_col=0, dtype=dtypes, skiprows=[1])
complete.insert(loc=1, column='Genus', value=complete['Taxon'].str.split('; ').str[1])

In [ ]:
# SECONDARY DATAFRAMES
# PERFORM CLR TRANSFORMATION IMMEDIATELY ON COMPLETE
complete_clr = pd.DataFrame(clr(complete.iloc[:,4:-1]+1, axis=0),index=complete.index, columns=complete.columns[4:-1]).pipe(
    complete.iloc[:,:3].join, **dict(how='inner'))
clr_10 = complete_clr.filter(regex=".*10.*")
clr_10 = pd.concat((clr_10,calc_pearson(clr_10.iloc[:,:-1], VOLTAGES)),axis=1,join='inner').pipe(complete_clr.iloc[:,:4].join, **dict(how='inner'))
clr_20_5 = complete.filter(regex=".*20_5.*")
clr_20_5 = pd.concat((clr_20_5,calc_pearson(clr_20_5.iloc[:,:-1], VOLTAGES)),axis=1,join='inner').pipe(complete_clr.iloc[:,:4].join, **dict(how='inner'))
clr_35 = complete_clr.filter(regex=".*35.*")
clr_35 = pd.concat((clr_35,calc_pearson(clr_35.iloc[:,:-1], VOLTAGES[:4])),axis=1,join='inner').pipe(complete_clr.iloc[:,:4].join, **dict(how='inner'))

In [ ]:
clr_10.sort_values("p_value")[['Genus','Taxon','r_value']].head(10)

In [ ]:
clr_20_5.sort_values("p_value")[['Genus','Taxon','r_value']].head(10)

In [ ]:
clr_35.sort_values("p_value")[['Genus','Taxon','r_value']].head(10)

In [ ]:
#FILTER FOR CONTROL SPECIES AND ELECTRO SPECIES USING THE CONTROL ELECTRODE
# Find control and experimental groups and filter on their indices 
complete_10 = complete.filter(regex=".*10m.*")
complete_20_5 = complete.filter(regex=".*20_5m.*")
complete_35 = complete.filter(regex=".*35m.*")

def filter_for_control(df, min_nonzero = 0, input_vars = VOLTAGES):
    """
    Params:
    df - The dataFrame to filter on the control condition. df must not be CLR-transformed.
    (those with less than 2 non-zero values and those in which the control CLR cell-counts are higher than the average voltage CLR cell-counts)
    Columns must be numeric and last column must represent control counts.
    input_vars - the sequence to correlate with the dfs
    
    Returns:
    (electro, control), where electro is all species beating the control conditon, control is all species failing to beat it
    """
    # Perform CLR transform on the df, we will use the CLR-transformed df to perform a filter later.
    df_clr = pd.DataFrame(clr(df+1, axis=0), index = df.index, columns = df.columns)
    # Calculate the Pearson R-correlations and store in df_pearson.
    df_pearson = calc_pearson(df_clr.iloc[:,:-1], input_vars)
    
    # Filter on raw sequence-counts having min_nonzero or more non-zero values across all electrodes at this depth.
    control = df[(df != 0).sum(axis = 1).values >= min_nonzero].iloc[:,:0]
    # Replace sequence-count data with CLR data from the CLR-transformed df
    control = df_clr.loc[control.index]
    # Filter on the CLR-value of the control electrode being greater than the average CLR-value of the voltage electrodes
    control = control[control.iloc[:,-1] >= control.iloc[:,:-1].mean(axis=1)]
    #Create the control_df to return by joining the control df with the first 4 columns of 'Complete_clr', which contain taxonomic data
    df_control = df_pearson.loc[control.index].pipe(
        # Join with df_clr to add values back in.
        df_clr.join, **dict(how='inner')).pipe(
        complete_clr.iloc[:,:4].join, **dict(how='inner'))
    #Create the electro_df by filtering df_clr to include only indices not included in control. Then join with complete_clr for taxonomic data.
    df_electro = df_pearson[~df_pearson.index.isin(control.index)].pipe(
        df_clr.join, **dict(how='inner')).pipe(
        complete_clr.iloc[:,:4].join, **dict(how='inner'))
    
    return (df_electro, df_control)

# Create filter groups
electro_10, control_10 = complete_10.pipe(filter_for_control)
electro_20_5, control_20_5 = complete_20_5.pipe(filter_for_control)
electro_35, control_35 = complete_35.pipe(filter_for_control, input_vars = VOLTAGES[:4])

In [ ]:
# GRAPHING %PPC TRENDS WITHIN FILTERED EXPERIMENTAL AND FILTERED CONTROL DATA SUBSETS
# Create three overlaying plots
fig, axes = plt.subplots(2, 1, figsize=(10, 10))

# Call the function for each subplot with different data/parameters
graph_ppp_line_subplot(electro_10, name="10m", ax=axes[0])
graph_ppp_line_subplot(electro_20_5, name="20.5m", ax=axes[0])
graph_ppp_line_subplot(electro_35, name="35m", ax=axes[0])

axes[0].axhline(y=0.5, color='red', linestyle='--', linewidth=1.5, label='50% threshold')
axes[0].set_ylim(0, 1)
axes[0].set_xlabel("alpha")
axes[0].set_ylabel("% Positive Pearson Correlation")
axes[0].legend(loc="upper center")
ax0 = axes[0].twinx()
ax0.set_ylabel('% Observations beating threshold')
ax0.set_ylim(0,1)
axes[0].set_title("%PPP vs alpha, filtered Experimental Group at 3 different depths")

# Call the function for each subplot with different data/parameters
graph_ppp_line_subplot(control_10, name="=10m", ax=axes[1])
graph_ppp_line_subplot(control_20_5, name="=20.5m", ax=axes[1])
graph_ppp_line_subplot(control_35, name="=35m", ax=axes[1])

axes[1].axhline(y=0.5, color='red', linestyle='--', linewidth=1.5, label='50% threshold')
axes[1].set_ylim(0, 1)
axes[1].set_xlabel("alpha")
axes[1].set_ylabel("% Positive Pearson Correlation")
axes[1].legend(loc="upper center")
ax1 = axes[1].twinx()
ax1.set_ylabel('% Observations beating threshold')
ax1.set_ylim(0,1)
axes[1].set_title("%PPP vs alpha, filtered Control Group at 3 different depths")

plt.tight_layout()
plt.show()

In [ ]:
#GRAPHING CLR ABUNDANCE OF 'CONTROL' AND 'ELECTRO' GROUPS AT DIFFRENT DEPTHS
def relative_abundance(filter_df,names=['included','excluded']):
    columns_of_interest = complete[filter_df.columns]

    included = columns_of_interest.loc[filter_df.index]
    excluded = columns_of_interest[~columns_of_interest.index.isin(filter_df.index)]

    abundance = pd.concat((included.sum(axis=0),excluded.sum(axis=0)),axis=1).transpose()
    rel_abundance = pd.DataFrame(clr(abundance+1, axis=0),index=names,columns=abundance.columns)
    return rel_abundance

rel_abundance_10 = relative_abundance(electro_10.iloc[:,4:9])
rel_abundance_20_5 = relative_abundance(electro_20_5.iloc[:,4:9])
rel_abundance_35 = relative_abundance(electro_35.iloc[:,4:8])


fig, axes = plt.subplots(1,3, figsize=(12,6))

axes[0].plot(VOLTAGES,rel_abundance_10.iloc[0], label='Electros 10m')
axes[0].plot(VOLTAGES,rel_abundance_10.iloc[1], label='Control 10m',color='gray')
axes[0].legend()
axes[1].plot(VOLTAGES,rel_abundance_20_5.iloc[0], label='Electros 20.5m')
axes[1].plot(VOLTAGES,rel_abundance_20_5.iloc[1], label='Control 20.5m',color='gray')
axes[1].legend()
axes[2].plot(VOLTAGES[:4],rel_abundance_35.iloc[0], label='Electros 35m')
axes[2].plot(VOLTAGES[:4],rel_abundance_35.iloc[1], label='Control 35m',color='gray')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
#AGGREGATE OBSERVATIONS BY GENUS AND CALCULAGE CLR VALUES AT EACH DEPTH
genus = pd.DataFrame(complete.groupby('Genus').size(),columns=['Num_Subspecies']).join(
    pd.DataFrame(complete.groupby('Genus').sum(numeric_only=True)))
genus_clr = pd.DataFrame(clr(genus.iloc[:,1:]+1,axis=0), index=genus.index, columns=genus.columns[1:])
genus_clr.insert(loc=0,column='Num_Subspecies',value=genus['Num_Subspecies'])
genus_clr.head()
genus_clr_10 = genus_clr.filter(regex=".*10m.*").iloc[:,:-1].pipe(calc_pearson, *[VOLTAGES]
            ).pipe(genus_clr.join, **dict(how='inner'))
genus_clr_20_5 = genus_clr.filter(regex=".*20_5m.*").iloc[:,:-1].pipe(calc_pearson, *[VOLTAGES]
            ).pipe(genus_clr.join, **dict(how='inner'))
genus_clr_35 = genus_clr.filter(regex=".*35m.*").iloc[:,:-1].pipe(calc_pearson, *[VOLTAGES[:4]]
            ).pipe(genus_clr.join, **dict(how='inner'))

In [ ]:
genus_clr_10[['Num_Subspecies','r_value','p_value']].sort_values('r_value').head()

In [ ]:
genus_clr_20_5[['Num_Subspecies','r_value','p_value']].sort_values('r_value').head()

In [ ]:
genus_clr_35[['Num_Subspecies','r_value','p_value']].sort_values('r_value').head()

In [ ]:
#GRAPHING RELATIVE CLR ABUNDANCE OF DIFFERENT GENUSES AT DIFFERENT DEPTHS
def graph_multiline(df,x,ax,title):
    for idx, row in df.iterrows():
        ax.plot(x,row.values,label=idx)
    ax.set_title(title)
    return ax

fig, axes = plt.subplots(1,3,figsize=(10,10))

for i, (df,title) in enumerate(zip([
    genus_clr_10.filter(regex=".*10m.*\\d.*"),
    genus_clr_20_5.filter(regex=".*20_5m.*\\d.*"),
    genus_clr_35.filter(regex=".*35m.*\\d.*")
],
['10m','20.5m','35m'])):
    graph_multiline(df, VOLTAGES[:len(df.columns)], axes[i], title)

plt.tight_layout()
plt.show()

In [ ]:
genus_clr_35.head()

In [ ]:
pd.concat((electro_10.iloc[:,4:9].sum(axis=0),electro_10.iloc[:,4:9].mean(axis=0)),axis=1).transpose().head()

In [ ]:
pd.DataFrame(.transpose()).head()